# EEGMamba: frozen encoder and linear classifier
Small educational screening: 5 training subjects and 2 validation subjects. PyTorch Mamba2 replaces native kernels; numerical parity with native implementation is unverified.


In [ ]:
import os,sys,json,random,time,ast,hashlib
from pathlib import Path
import numpy as np,pandas as pd,torch
from torch import nn
from scipy.signal import resample_poly
from sklearn.metrics import balanced_accuracy_score,accuracy_score
root=Path(r'D:\EEG Experiment Artifacts\2026-11-02');source=root/'mamba_sources'/'EEGMamba';weights=root/'mamba_sources'/'EEGMamba_weights'/'pretrained_EEGMamba.pth'
sys.path[:0]=[str(root/'pretrained_experiments'/'reve_dependencies'),str(root/'pretrained_experiments'/'dependencies')]
from einops import rearrange
from transformers import Mamba2Config
from transformers.models.mamba2.modeling_mamba2 import Mamba2Mixer
seed=42;random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed);torch.set_num_threads(4)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu');assert device.type=='cuda'
protocol=json.loads((root/'binary_de_psd'/'protocol.json').read_text(encoding='utf8'));full_split=protocol['splits'][0];split={'train':full_split['train'][:5],'val':full_split['val'][:2],'test':full_split['test']}
subjects=np.repeat(np.arange(123),24);video_ids=np.r_[0:12,16:28];labels=np.tile(np.r_[np.zeros(12,dtype=np.int64),np.ones(12,dtype=np.int64)],123)
indices={k:np.flatnonzero(np.isin(subjects,split[k])) for k in ('train','val')}
assert not set(split['train'])&set(split['val']) and not set(split['test'])&set(split['train']+split['val'])
raw=np.load(root/'cache'/'raw_windows_30ch.npy',mmap_mode='r');assert raw.shape==(3444,30,30,250)
code=(source/'models'/'eegmamba.py').read_text(encoding='utf8');node=next(x for x in ast.parse(code).body if isinstance(x,ast.ClassDef) and x.name=='PatchEmbedding');exec(compile(ast.Module(body=[node],type_ignores=[]),str(source/'models'/'eegmamba.py'),'exec'))
config=Mamba2Config(hidden_size=200,expand=2,num_heads=8,head_dim=50,state_size=64,n_groups=1,layer_norm_epsilon=1e-5,chunk_size=32)
class Layer(nn.Module):
 def __init__(self,i):
  super().__init__();self.norm=nn.RMSNorm(200,eps=1e-5);self.mixer=Mamba2Mixer(config,layer_idx=i)
class Encoder(nn.Module):
 def __init__(self):
  super().__init__();self.layers=nn.ModuleList([Layer(i) for i in range(12)]);self.norm_f=nn.RMSNorm(200,eps=1e-5)
 def forward(self,x):
  residual=None
  for layer in self.layers:
   residual=x if residual is None else x+residual
   x=layer.mixer(layer.norm(residual));x=x.flip(1);residual=residual.flip(1)
  return self.norm_f(x+residual)
class EEGMamba(nn.Module):
 def __init__(self):
  super().__init__();self.patch_embedding=PatchEmbedding(200,200,200,30);self.encoder=Encoder();self.proj_out=nn.Sequential(nn.Linear(200,200))
 def forward(self,x):
  batch,ch,seq,_=x.shape;h=self.patch_embedding(x,mask=None);h=self.encoder(rearrange(h,'b c t d -> b (c t) d'));return rearrange(h,'b (c t) d -> b c t d',c=ch,t=seq)
model=EEGMamba();state=torch.load(weights,map_location='cpu',weights_only=True);model.load_state_dict(state,strict=True);model=model.to(device).eval()
for param in model.parameters():param.requires_grad_(False)
print('DEVICE',torch.cuda.get_device_name(0),'STRICT WEIGHT LOAD OK','PARAMETERS',sum(x.numel() for x in model.parameters()),flush=True)


In [ ]:
# The author network uses Mamba2; transformers' PyTorch implementation replaces its CUDA-only operators.
# Mean pooling applies to learned encoder tokens; raw EEG is not averaged across channels.
output=root/'mamba_experiments'/'eegmamba_frozen';output.mkdir(parents=True,exist_ok=True)
manifest={'model':'EEGMamba frozen','implementation':'author patch embedding and residual/reverse order; Transformers PyTorch Mamba2 mixer','weights_sha256':hashlib.sha256(weights.read_bytes()).hexdigest(),'fold':0,'split':split,'seed':seed,'preprocessing':'per-trial/channel zscore existing cache; 250 to 200 Hz; three ten-second windows','pooling':'mean over encoder tokens','notes':'native CUDA Mamba2 implementation unavailable; numerical parity with native implementation not yet established'}
file=output/'config.json'
if file.exists():assert json.loads(file.read_text())==manifest
else:file.write_text(json.dumps(manifest,indent=2),encoding='utf8')
for split_name in ['train','val']:
 ids=indices[split_name];path=output/(split_name+'_embeddings.npy');progress=output/(split_name+'_progress.json')
 if path.exists():emb=np.lib.format.open_memmap(path,mode='r+');done=json.loads(progress.read_text())['completed']
 else:emb=np.lib.format.open_memmap(path,mode='w+',dtype=np.float32,shape=(len(ids),3,200));done=0;progress.write_text(json.dumps({'completed':0}))
 assert emb.shape==(len(ids),3,200)
 with torch.inference_mode():
  for i in range(done,len(ids)):
   j=int(ids[i]);raw_idx=int(subjects[j])*28+int(video_ids[j%24]);x=np.asarray(raw[raw_idx],dtype=np.float32).transpose(1,0,2).reshape(30,7500);x=resample_poly(x,4,5,axis=-1).reshape(30,3,10,200).transpose(1,0,2,3).copy()
   for k in range(3):emb[i,k]=model(torch.from_numpy(x[k:k+1]).to(device)).mean(dim=(1,2)).cpu().numpy()[0]
   if (i+1)%5==0 or i+1==len(ids):emb.flush();progress.write_text(json.dumps({'completed':i+1}));print('EMBEDDINGS',split_name,i+1,'/',len(ids),flush=True)
 del emb


In [ ]:
train=np.load(output/'train_embeddings.npy').reshape(-1,200);val=np.load(output/'val_embeddings.npy').reshape(-1,200);y_train=np.repeat(labels[indices['train']],3);y_val=np.repeat(labels[indices['val']],3)
mean=train.mean(0);std=np.maximum(train.std(0),1e-6);np.savez(output/'scaler.npz',mean=mean,std=std)
head=nn.Linear(200,2).to(device);optimizer=torch.optim.AdamW(head.parameters(),lr=1e-3,weight_decay=1e-3);rng=np.random.default_rng(seed);history=[];best=-1
for epoch in range(1,4):
 head.train();order=rng.permutation(len(train));losses=[]
 for batch in np.array_split(order,int(np.ceil(len(order)/256))):
  x=torch.from_numpy((train[batch]-mean)/std).to(device);y=torch.from_numpy(y_train[batch]).to(device);optimizer.zero_grad(set_to_none=True);loss=nn.functional.cross_entropy(head(x),y);loss.backward();optimizer.step();losses.append(loss.item())
 head.eval()
 with torch.no_grad():pred=head(torch.from_numpy((val-mean)/std).to(device)).argmax(1).cpu().numpy()
 score=balanced_accuracy_score(y_val,pred);history.append({'epoch':epoch,'loss':float(np.mean(losses)),'validation_accuracy':float(accuracy_score(y_val,pred)),'validation_balanced_accuracy':float(score)});pd.DataFrame(history).to_csv(output/'history.csv',index=False)
 if score>best:best=score;torch.save({'model':head.state_dict(),'epoch':epoch},output/'best_model.pt');np.savez_compressed(output/'validation_predictions.npz',predictions=pred,labels=y_val)
 print('EPOCH',epoch,'VALIDATION BALANCED ACCURACY',score,flush=True)
(output/'metrics.json').write_text(json.dumps({'best_validation_balanced_accuracy':best,'epochs':3,'fold':0},indent=2),encoding='utf8');print('COMPLETE',flush=True)
